*0.1 Python for GenAI*

# Pydantic v2 models

**The situation.** Your web service has a signup endpoint. A mobile app sends `{"name": "Priya", "age": "thirty", "email": "priya"}`. The code stores it as it is. Two weeks later a birthday report crashes on `age + 1`, and a mail job bounces on the email address — two bugs, far from where the bad data entered, with no record of which request caused them.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The bad version: trust the input.** This endpoint takes whatever it is given and saves it.

In [2]:
from fastapi import FastAPI

app = FastAPI()
saved_customers = []


@app.post("/signup-unchecked")
async def signup_unchecked(customer: dict) -> dict:
    saved_customers.append(customer)  # nothing is checked; anything goes into the database
    return {"saved": customer}


print("endpoint registered: /signup-unchecked")

endpoint registered: /signup-unchecked


**The good version: describe the shape once.** A Pydantic model lists the fields and their types. FastAPI checks every request against it *before* your code runs. Wrong data never gets in; the app that sent it gets a precise error back.

In [3]:
from pydantic import BaseModel, EmailStr, Field


class Customer(BaseModel):
    name: str = Field(min_length=1, max_length=80)
    age: int = Field(ge=0, le=150)
    email: EmailStr  # must look like an email address


@app.post("/signup")
async def signup(customer: Customer) -> dict:
    saved_customers.append(customer.model_dump())  # by now every field is the right type
    return {"saved": customer.model_dump()}


print("endpoint registered: /signup")

endpoint registered: /signup


**Send the mobile app's bad request to both.**

In [4]:
from fastapi.testclient import TestClient

bad_request = {"name": "Priya", "age": "thirty", "email": "priya"}

with TestClient(app) as web:
    unchecked = web.post("/signup-unchecked", json=bad_request)
    checked = web.post("/signup", json=bad_request)

print("unchecked:", unchecked.status_code, unchecked.json())
print("checked:  ", checked.status_code)
for problem in checked.json()["detail"]:
    print("   field", problem["loc"][-1], "→", problem["msg"][:70])
assert unchecked.status_code == 200 and checked.status_code == 422

unchecked: 200 {'saved': {'name': 'Priya', 'age': 'thirty', 'email': 'priya'}}
checked:   422
   field age → Input should be a valid integer, unable to parse string as an integer
   field email → value is not a valid email address: An email address must have an @-si


**Reading the output.** The unchecked endpoint said 200 OK and stored `"thirty"` as an age. The checked endpoint refused with 422 and named both problems, field by field. The bug is caught at the door, at the moment it happens, with the request that caused it.

**And a good request.** Notice `"age": "30"` — text — becomes the number 30. Pydantic converts when it is safe.

In [5]:
with TestClient(app) as web:
    good = web.post("/signup", json={"name": "Priya", "age": "30", "email": "priya@example.com"})
print(good.status_code, good.json())
assert good.json()["saved"]["age"] == 30

200 {'saved': {'name': 'Priya', 'age': 30, 'email': 'priya@example.com'}}


**The rule to remember.** Every place data enters your system — a web request, a message from a queue, a file, an answer from the model — gets a Pydantic model. Inside the system, after that door, the data can be trusted.

| Use it when | Don't when | Instead use |
|---|---|---|
| data crosses a boundary: web requests, queue messages, config, model output | passing trusted data around inside the program, millions of times | plain dataclasses for internal records (a later item) |

**Watch out**
- Unknown fields are silently dropped by default. Add `model_config = ConfigDict(extra="forbid")` so a misspelled field is an error, not a silent loss.
- `"30"` → `30` is helpful for forms and dangerous for money. Use `strict=True` on models where conversion is not acceptable.
- A field like `tags: list = []` shares one list between all objects. Write `Field(default_factory=list)`.